# Deriving Topics from Comments

In [ ]:
from datetime import datetime
from turftopic import KeyNMF
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from turftopic.analyzers import LLMAnalyzer

## Ignore this rn but wanna do something like this

In [17]:
model.plot_topics_over_time()

#### For multiple, not tested yet just fyi

In [ ]:
# Folder containing your CSVs
csv_folder = Path("/Users/au728638/Library/CloudStorage/OneDrive-Aarhusuniversitet/Desktop/4. Other Projects/Resident_Evil/raw_reviews")
text_column = "review"  # change to your column name

# Reuse the encoder across files (loading it is the slow part)
results = {}
for csv_path in csv_folder.glob("*.csv"):
    df = pd.read_csv(csv_path)
    docs = df[text_column].dropna().astype(str).tolist()

    if len(docs) < 10:
        print(f"Skipping {csv_path.name}: only {len(docs)} docs")
        continue

    model = KeyNMF(10, encoder=encoder)
    model.fit(docs)
    results[csv_path.name] = model
    print(f"Fitted {csv_path.name}")

# Inspect topics for each file
for name, model in results.items():
    print(f"\n=== {name} ===")
    model.print_topics()

## Topic Modeling with KeyNMF (pooled RE2r/RE3r)

In [4]:
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
csv_paths = {
    "re2r": Path("/Users/au728638/Library/CloudStorage/OneDrive-Aarhusuniversitet/Desktop/4. Other Projects/Resident_Evil/raw_reviews/resident_evil_2_remake.csv"),
    "re3r": Path("/Users/au728638/Library/CloudStorage/OneDrive-Aarhusuniversitet/Desktop/4. Other Projects/Resident_Evil/raw_reviews/resident_evil_3_remake.csv"),
}
text_column = "review"
date_column = "date_created"

dfs = []
for game, path in csv_paths.items():
    game_df = pd.read_csv(path)
    game_df["game"] = game
    dfs.append(game_df)
df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=[text_column])

docs = df[text_column].astype(str).tolist()
game_labels = df["game"].tolist()  # keep alongside docs/embeddings for per-game breakdowns later
hover_text = [f"[{g}] {doc[:150]}" for g, doc in zip(game_labels, docs)]  # shown when hovering a point in plot_clusters_datamapplot

embeddings = encoder.encode(docs, show_progress_bar=True)

Batches:   0%|          | 0/3900 [00:00<?, ?it/s]

## Mentions of other games

In [22]:
import re
from rapidfuzz import fuzz

# RQ1: how often are other mainline/remake titles referenced in RE2r/RE3r reviews?
# Heuristic keyword/regex detector — not exhaustive NLP entity recognition,
# but far more precise and reportable than reading it off topic keywords.
TITLE_PATTERNS = {
    "re1": r"\bre\s?1\b|\bresident evil 1\b",
    "re0": r"\bre\s?0\b|\bresident evil 0\b|\bre zero\b",
    "re2": r"\bre\s?2\b|\bresident evil 2\b",
    "re3": r"\bre\s?3\b|\bresident evil 3\b|\bnemesis\b",
    "re4": r"\bre\s?4\b|\bresident evil 4\b",
    "re5": r"\bre\s?5\b|\bresident evil 5\b",
    "re6": r"\bre\s?6\b|\bresident evil 6\b",
    "re7": r"\bre\s?7\b|\bresident evil 7\b|\bbiohazard\b",
    "village": r"\bre\s?8\b|\bresident evil village\b|\bvillage\b",
    "revelations": r"\brevelations\b",
    "requiem": r"\brequiem\b",
    # "Raccoon City" alone is the games' fictional SETTING (both RE2 and RE3
    # take place there) — only "operation raccoon city" actually refers to
    # the separate 2012 spin-off title, so the bare phrase is deliberately
    # NOT matched here to avoid flagging ordinary setting mentions.
    "raccoon_city": r"\boperation raccoon city\b",
}
# A game reviewing itself (e.g. "re2" in an re2r review) isn't a cross-title
# reference, so it's excluded per-row rather than counted.
SELF_TITLE = {"re2r": "re2", "re3r": "re3"}

# Fuzzy backstop for misspellings, split into two parts on purpose:
# "resident evil <N>" phrases only differ from each other by the trailing
# digit (~1 char out of ~15), so fuzzy-matching the whole phrase confuses
# N=1 with N=4 with N=7, etc. Instead: fuzzy-match "resident evil" only,
# then require the next token's digit to match exactly.
FUZZY_THRESHOLD = 85
NUMBER_TO_TITLE = {"0": "re0", "zero": "re0", "1": "re1", "2": "re2", "3": "re3",
                    "4": "re4", "5": "re5", "6": "re6", "7": "re7", "village": "village"}

# Standalone unique words/phrases — safe to fuzzy-match directly since they
# aren't near-duplicate templates like the numbered titles are.
FUZZY_PHRASES = {
    "re7": ["biohazard"],
    "re3": ["nemesis"],
    "revelations": ["revelations"],
    "requiem": ["requiem"],
    "raccoon_city": ["operation raccoon city"],  # not bare "raccoon city" — see note above
}


def fuzzy_contains(text_l: str, phrase: str) -> bool:
    words = text_l.split()
    n = len(phrase.split())
    return any(
        fuzz.ratio(" ".join(words[i : i + n]), phrase) >= FUZZY_THRESHOLD
        for i in range(len(words) - n + 1)
    )


def fuzzy_resident_evil_number(text_l: str) -> set:
    words = text_l.split()
    hits = set()
    for i in range(len(words) - 1):
        window = f"{words[i]} {words[i + 1]}"
        if fuzz.ratio(window, "resident evil") >= FUZZY_THRESHOLD and i + 2 < len(words):
            nxt = words[i + 2].strip(".,!?")
            if nxt in NUMBER_TO_TITLE:
                hits.add(NUMBER_TO_TITLE[nxt])
    return hits


def find_other_title_mentions(text: str, game: str) -> list[str]:
    text_l = text.lower()
    self_key = SELF_TITLE.get(game)

    hits = {
        name
        for name, pattern in TITLE_PATTERNS.items()
        if name != self_key and re.search(pattern, text_l)
    }
    hits |= fuzzy_resident_evil_number(text_l) - {self_key}
    hits |= {
        name
        for name, phrases in FUZZY_PHRASES.items()
        if name != self_key and any(fuzzy_contains(text_l, p) for p in phrases)
    }
    return sorted(hits)


df["other_title_mentions"] = [
    find_other_title_mentions(text, game)
    for text, game in zip(df["review"], df["game"])
]
df["mentions_other_title"] = df["other_title_mentions"].apply(len) > 0

pct_mentioning_other_title = df.groupby("game")["mentions_other_title"].mean() * 100
print("% of reviews referencing another RE title:")
print(pct_mentioning_other_title)
print()

# Which other titles get referenced most, per game
mention_counts = (
    df.explode("other_title_mentions")
    .dropna(subset=["other_title_mentions"])
    .groupby(["game", "other_title_mentions"])
    .size()
    .unstack(fill_value=0)
)
mention_counts

% of reviews referencing another RE title:
game
re2r     7.726104
re3r    23.767830
Name: mentions_other_title, dtype: float64



other_title_mentions,raccoon_city,re0,re1,re2,re3,re4,re5,re6,re7,requiem,revelations,village
game,,,,,,,,,,,,
re2r,23,191,767,0,2581,1925,300,295,1300,325,150,313
re3r,36,83,441,9717,0,1123,225,345,490,141,86,285


In [18]:
# A simple, direct grounding stat for "different levels of criticism" ahead
# of the topic-level comparison built later in this notebook.
sentiment_by_game = df.groupby("game")["sentiment"].agg(["mean", "count"]).rename(
    columns={"mean": "pct_positive", "count": "n_reviews"}
)
sentiment_by_game["pct_positive"] *= 100
sentiment_by_game

,pct_positive,n_reviews
game,,
re2r,96.855888,80118
re3r,82.117025,44657


In [23]:
# find reviewers who reviewed BOTH games, and pair up their reviews for
# qualitative reading — e.g. did the same person love one remake and hate the other?
author_game_counts = df.groupby("author_number")["game"].nunique()
both_games_authors = author_game_counts[author_game_counts > 1].index

re2r_by_author = df[(df["game"] == "re2r") & (df["author_number"].isin(both_games_authors))]
re3r_by_author = df[(df["game"] == "re3r") & (df["author_number"].isin(both_games_authors))]

paired_reviews = re2r_by_author.merge(
    re3r_by_author,
    on="author_number",
    suffixes=("_re2r", "_re3r"),
)[[
    "author_number",
    "review_re2r", "sentiment_re2r", "date_created_re2r",
    "review_re3r", "sentiment_re3r", "date_created_re3r",
]]

paired_reviews["sentiment_flipped"] = paired_reviews["sentiment_re2r"] != paired_reviews["sentiment_re3r"]
pos_to_neg = ((paired_reviews["sentiment_re2r"] == 1) & (paired_reviews["sentiment_re3r"] == 0)).sum()
neg_to_pos = ((paired_reviews["sentiment_re2r"] == 0) & (paired_reviews["sentiment_re3r"] == 1)).sum()
print(
    f"{len(paired_reviews)} reviewers reviewed both games "
    f"({paired_reviews['sentiment_flipped'].sum()} with flipped sentiment: "
    f"{pos_to_neg} liked RE2r but disliked RE3r, "
    f"{neg_to_pos} disliked RE2r but liked RE3r)."
)

# Sort so the most qualitatively interesting cases (opposite sentiment) come first
paired_reviews.sort_values("sentiment_flipped", ascending=False)

19060 reviewers reviewed both games (2587 with flipped sentiment: 2427 liked RE2r but disliked RE3r, 160 disliked RE2r but liked RE3r).


,author_number,review_re2r,sentiment_re2r,date_created_re2r,review_re3r,sentiment_re3r,date_created_re3r,sentiment_flipped
10074,author_036886,Most of the praise I give for this game goes t...,1,2023-01-01,My overall thoughts about this game are meh. T...,0,2023-01-11,True
11941,author_044174,Love it!,1,2021-11-24,The first thing I have to explain here is that...,0,2022-04-19,True
11935,author_044149,"""Resident Evil 2"" has everything we expect fro...",1,2021-11-25,A sweet that tasted bitter after years! \n \nR...,0,2024-05-14,True
16329,author_074828,That guy's a maniac! WHY'DHEBITEME?\n\nThat ma...,1,2019-11-28,Disappointing in almost every way. Music was O...,0,2020-04-07,True
3041,author_011515,"My favourite in the series so far, Leon, Clair...",1,2026-02-21,"I shell of its former self, almost unreconizab...",0,2026-02-21,True
...,...,...,...,...,...,...,...,...
6702,author_024406,It's not an easy game. Not an easy work to aim...,1,2024-06-10,"It's a good game however, as ppl said and it's...",1,2026-03-25,False
6703,author_024407,"""Damn... Shoulda packed my parka."" \n \n10/10 ...",1,2024-06-10,it's fun but like only 2 minutes long,1,2025-08-19,False
6704,author_024412,Great game one of the best Resident Evil Games...,1,2024-06-09,Pretty good but I prefer all other titles RE4 ...,1,2024-06-09,False
6705,author_024420,the kiss>>>,1,2024-06-09,---{ Graphics }---\n☐ You forget what reality ...,1,2024-04-28,False


In [7]:
# n_components picked as a starting point for a paper-scale topic table;
# adjust after checking print_topics() below for redundant/incoherent topics.
model = KeyNMF(30, encoder=encoder)
topic_data = model.prepare_topic_data(docs, embeddings=embeddings)

Output()

[18:19:50] Keyword extraction done.                                                                   ]8;id=357977;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/models/keynmf.py\keynmf.py]8;;\:]8;id=426284;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/models/keynmf.py#448\448]8;;\

[18:19:53] Model fitting done.                                                                        ]8;id=243712;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/models/keynmf.py\keynmf.py]8;;\:]8;id=518685;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/models/keynmf.py#459\459]8;;\

In [8]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        0 │ game, playing, awesome, like, time, puzzles, 2019, nice, fantastic, recommend                        │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ remake, resident, perfect, excellent, new, make, fantastic, veronica, masterpiece, series            │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ horror, survival, experience, terrifying, genre, action, combat, fear, scares, intense               │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ re2, better, dlc, re1, compared, re7, like, different, enjoyed, liked                                │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ evil, resident, residents, best, feel, raccoon, fan, experience, terrifying, overall                 │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ fun, puzzles, playing, like, challenging, spooky, pretty, run, really, recommend                     │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        6 │ nemesis, fight, like, fights, enemies, carlos, raccoon, enemy, inferno, re2r                         │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        7 │ good, like, overall, bad, pretty, really, graphics, better, just, recommend                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │ capcom, drm, review, dlc, did, re7, remaking, resistance, enigma, gaming                             │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        9 │ leon, claire, kennedy, ada, love, redfield, tyrant, hot, finished, mr                                │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │ games, favorite, best, like, time, playing, gaming, drm, favourite, puzzles                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       11 │ re3, re2r, resistance, re3r, review, re1, replayability, re6, compared, raccoon                      │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       12 │ gameplay, graphics, annoying, difficulty, bad, overall, difficult, brain, paint, decent              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       13 │ zombies, kill, ammo, enemies, puzzles, enemy, combat, killing, terrifying, shoot                     │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       14 │ original, better, version, different, new, nostalgia, ps1, remaster, cut, resident                   │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       15 │ scary, mr, puzzles, scared, scares, tyrant, big, creepy, annoying, man                               │
├──────────┼────────────────────────────────────────────

In [ ]:
# Built once and reused across re-fits — loading the local LLM is the slow part,
# same reasoning as reusing `encoder` above.
# Qwen2.5-3B-Instruct: same size class as the previous default (SmolLM3-3B) so
# memory/speed stay comparable, but noticeably stronger at instruction-following.
analyzer = LLMAnalyzer(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    context="Analyze topics from Steam user reviews of Resident Evil 2 Remake and Resident Evil 3 Remake. Use information regarding the name to build labels and descriptions that are concise, descriptive, and suitable for a table of topics. Avoid generic labels like 'Gameplay' or 'Story'.",
    device="mps",
)

In [14]:
analysis_result = model.analyze_topics(analyzer, use_documents=True)
model.print_topics()

Output()

[21:03:27] Topic names generated.                                                                       ]8;id=751337;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/analyzers/base.py\base.py]8;;\:]8;id=944491;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/analyzers/base.py#229\229]8;;\

Output()

[21:04:36] Topic descriptions generated.                                                                ]8;id=753382;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/analyzers/base.py\base.py]8;;\:]8;id=134966;file:///Users/au728638/venvs/resident_evil/lib/python3.12/site-packages/turftopic/analyzers/base.py#234\234]8;;\

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Topic Name                      ┃ Topic Descriptions               ┃ Highest Ranking                 ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        0 │ 2019 Puzzle Games               │ This topic revolves around       │ game, playing, awesome, like,   │
│          │ Recommendation                  │ positive experiences with games, │ time, puzzles, 2019, nice,      │
│          │                                 │ particularly those that involve  │ fantastic, recommend            │
│          │                                 │ solving puzzles, and individuals │                                 │
│          │                                 │ expressing their enjoyment from  │                                 │
│          │                                 │ 2019 onwards. The games are      │                                 │
│          │                                 │ frequently recommended due to    │                                 │
│          │                                 │ their engaging and enjoyable     │                                 │
│          │                                 │ nature.                          │                                 │
├──────────┼─────────────────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│        1 │ Remake Series: Veronica's       │ The topic revolves around a      │ remake, resident, perfect,      │
│          │ Perfect Masterpiece             │ remake or adaptation of a series │ excellent, new, make,           │
│          │                                 │ that aims to surpass the         │ fantastic, veronica,            │
│          │                                 │ original, aiming to be an        │ masterpiece, series             │
│          │                                 │ excellent, perfect, and          │                                 │
│          │                                 │ fantastic version featuring a    │                                 │
│          │                                 │ character named Veronica, with   │                                 │
│          │                                 │ the goal of creating a           │                                 │
│          │                                 │ masterpiece.                     │                                 │
├──────────┼─────────────────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│        2 │ Horror Survival Experience      │ This topic revolves around       │ horror, survival, experience,   │
│          │                                 │ narratives within the horror     │ terrifying, genre, action,      │
│          │                                 │ genre that focus on survival     │ combat, fear, scares, intense   │
│          │                                 │ scenarios, often featuring       │                                 │
│          │                                 │ intense and terrifying           │                                 │
│          │                                 │ situations. It includes elements │                                 │
│          │                                 │ of action and combat, designed   │                                 │
│          │                                 │ to evoke fear and provide a      │                                 │
│          │                                 │ thrilling, albeit frightening,   │                                 │
│          │                                 │ experience for the audience.     │                                 │
├──────────┼─────────────────────────────────┼──────────────────────────────────┼─────────────────────────────────┤
│        3 │ Video Game Reboot Comparisons   │ The topic

In [16]:
# KeyNMF already gives every review a soft weight on every topic — no extra
# scoring step needed. document_topic_matrix has shape (n_documents, n_topics),
# in the same row order as docs/game_labels.
dtm = topic_data.document_topic_matrix

# Per-review topic scores (row i = review i's weight across all topics)
review_topic_scores = pd.DataFrame(dtm, columns=model.topic_names)
review_topic_scores.insert(0, "game", game_labels)
review_topic_scores.insert(1, "review", docs)

# Per-game topic prevalence — the comparison the paper's second leading
# question is asking for (RE2r vs RE3r topic differences)
per_game_topic_prevalence = review_topic_scores.drop(columns="review").groupby("game").mean()
per_game_topic_prevalence

,2019 Puzzle Games Recommendation,Remake Series: Veronica's Perfect Masterpiece,Horror Survival Experience,Video Game Reboot Comparisons,Raccoon Resident Fan Experience,Spooky Puzzle Adventure,Raccoon Enemy Matches: Carlos vs. Inferno & RE2R,Game Graphics Review,Resistance Reimagined DLC Review,Cold War Era Political Scandals,...,Amazing Graphics Experience,Playtime Recommendations for Claire,Zombie Apocalypse Combat Essentials,Adventure Story with Graphics,Movie Review: Mr. Graphics 10/11 - Like Ada Wong's Scary Moments,Summer Sale Highlights: Sweet Deals & Pretty Finds,Movie/TV Series Reviews,Classic Survival Remasters: Nostalgia & Reimagining\n\nReplayability & Fantastical Modern Takes,Video Game Series Enjoyment,Gameplay Analysis and Replayability
game,,,,,,,,,,,,,,,,,,,,,
re2r,0.012117,0.007275,0.005074,0.002595,0.004333,0.003346,0.000477,0.003916,0.003056,0.003438,...,0.003137,0.003802,0.003113,0.003300,0.003012,0.000268,0.001764,0.002476,0.003339,0.002652
re3r,0.012902,0.009209,0.002180,0.008512,0.004400,0.005303,0.009010,0.005265,0.003179,0.000210,...,0.001512,0.003631,0.001953,0.003311,0.001984,0.006176,0.002059,0.001273,0.001880,0.004326


In [24]:
import numpy as np
from statsmodels.stats.multitest import multipletests

# Permutation test on the difference in means, rather than Mann-Whitney U +
# rank-biserial correlation — this attaches significance directly to the same
# re2r_mean/re3r_mean numbers already being reported, instead of a separate
# abstract statistic. Nonparametric like Mann-Whitney (no normality assumed).
N_PERMUTATIONS = 10_000
rng = np.random.default_rng(0)

dtm = review_topic_scores[model.topic_names].to_numpy()
is_re2r = (review_topic_scores["game"] == "re2r").to_numpy()

# mean_diff = re3r_mean - re2r_mean (positive = more prevalent in RE3r)
observed_diff = dtm[~is_re2r].mean(axis=0) - dtm[is_re2r].mean(axis=0)

idx = np.arange(len(dtm))
null_diffs = np.empty((N_PERMUTATIONS, dtm.shape[1]))
for i in range(N_PERMUTATIONS):
    rng.shuffle(idx)
    shuffled_mask = is_re2r[idx]
    null_diffs[i] = dtm[~shuffled_mask].mean(axis=0) - dtm[shuffled_mask].mean(axis=0)

p_values = (np.abs(null_diffs) >= np.abs(observed_diff)).mean(axis=0)
# floor at 1/(N+1) since a finite number of permutations can't distinguish p=0
# from "just smaller than we could measure"
p_values = np.maximum(p_values, 1 / (N_PERMUTATIONS + 1))

topic_diff = pd.DataFrame({
    "topic": model.topic_names,
    "re2r_mean": dtm[is_re2r].mean(axis=0),
    "re3r_mean": dtm[~is_re2r].mean(axis=0),
    "mean_diff": observed_diff,
    "p_value": p_values,
})
# Benjamini-Hochberg FDR correction across all 30 topics tested at once.
topic_diff["significant_fdr"] = multipletests(topic_diff["p_value"], method="fdr_bh")[0]
topic_diff.sort_values("mean_diff", key=abs, ascending=False)

,topic,re2r_mean,re3r_mean,mean_diff,p_value,significant_fdr
6,Raccoon Enemy Matches: Carlos vs. Inferno & RE2R,0.000477,0.009010,0.008533,0.0001,True
3,Video Game Reboot Comparisons,0.002595,0.008512,0.005917,0.0001,True
25,Summer Sale Highlights: Sweet Deals & Pretty F...,0.000268,0.006176,0.005908,0.0001,True
17,Romantic Sandwiches\n\n(Descriptive phrase bas...,0.000178,0.005915,0.005737,0.0001,True
19,Game DLC Pricing Recommendations,0.001494,0.006647,0.005153,0.0001,True
11,Replayability Review Comparisons,0.001048,0.005676,0.004628,0.0001,True
9,Cold War Era Political Scandals,0.003438,0.000210,-0.003228,0.0001,True
2,Horror Survival Experience,0.005074,0.002180,-0.002895,0.0001,True
5,Spooky Puzzle Adventure,0.003346,0.005303,0.001958,0.0001,True
1,Remake Series: Veronica's Perfect Masterpiece,0.007275,0.009209,0.001935,0.0001,True


## Summary of findings

### RQ1 — References to other titles
- RE3r reviews reference another RE title 3x more often than RE2r (23.8% vs 7.7%).
- RE3r reviewers overwhelmingly reference RE2 (9717 mentions, far more than any other title in either game), RE3r is constantly measured against its sibling
- RE2r reviewers mostly reference RE3 (2581) and RE4 (1925).

### RQ2 — Topic and sentiment comparison
- Overall sentiment: RE2r 96.9% positive vs RE3r 82.1% positive, an unambiguous gap.
- More prevalent in RE3r: Carlos/Nemesis/"Inferno" enemy fights (largest topic gap by far); reboot/version comparisons (re1/re2/re7); sale/price/DLC discussion; replayability comparisons.
- More prevalent in RE2r: a character-focused topic (Leon, Claire, Ada, Redfield, "tyrant" mislabeled by the LLM as "Cold War Era Political Scandals,"; general horror/survival framing.
- A few topics (Resistance/DLC, general fan-experience, generic story/graphics praise) showed no meaningful difference between games.

### RQ3 — Same reviewer, both games
- 19,060 reviewers rated both games.
- 2,587 (13.6%) flipped sentiment between games, and it's heavily one-directional: 2,427 liked RE2r but disliked RE3r, vs only 160 the reverse (~15:1 ratio).
- This suggests RE3r's criticism partly comes from people who already liked the sibling remake turning on it specifically, not two separate populations of critics and fans.
- might want to limit analyses in RQ2 to just people who reviewed both?